# ViSceT5 — Finetune + Ablation · nhánh `exp/ablation-drop-ocr` · Colab A100

Không pretrain, không visual search, không MRA. Mỗi lần chạy = **một cấu hình**, chọn bằng 3 công tắc bật/tắt module ở cell **Cấu hình**:

| Công tắc | `True` | `False` |
|---|---|---|
| `USE_QAVIT` | đưa câu hỏi vào CLIP (QA-ViT) | CLIP trích xuất ảnh thường (không câu hỏi) |
| `USE_SCESPAVIS` | OCR qua SceSpaVis (ConsFormer + char + box + det/rec) | OCR text-only |
| `USE_OCRAUG` | nối thêm bản OCR đã sửa (OCR-Aug) | chỉ OCR gốc |

**ViSceT5 (kết quả chính):** `USE_QAVIT=False, USE_SCESPAVIS=True, USE_OCRAUG=True`. Bảng ablation: chạy lại notebook với các tổ hợp khác (vd tắt hết = baseline).

Cố định cho mọi cấu hình: 5 epoch, batch 4×2, LR 3e-5, chọn model theo EM trên val, beam 4. Thư mục output và repo HF tự đặt tên theo module đang bật nên các lần chạy không đè nhau.

## 1. Clone đúng nhánh

In [ ]:
import os, sys
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
REPO_DIR = os.path.join(WORK_DIR, "ViSceT5")
BRANCH = "exp/ablation-drop-ocr"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Kussssssss/ViSceT5.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin
!git checkout {BRANCH}
!git reset --hard origin/{BRANCH}
!git log --oneline -3
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Bảo đảm đã có bản sửa OCR-Aug ("<pad>" không bị "sửa" thành "ipad")
assert "_specials" in open("data/collator.py", encoding="utf-8").read(), "Thiếu bản sửa OCR-Aug — kiểm tra lại nhánh!"
print("✅ Nhánh", BRANCH, "có bản sửa OCR-Aug.")

## 2. Cài môi trường (transformers 4.45.2)

In [ ]:
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap

In [ ]:
import torch, transformers
print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("Transformers", transformers.__version__)
assert transformers.__version__.startswith("4.45"), "Cần transformers 4.45.x"

## 3. Đăng nhập Hugging Face (tự push checkpoint mỗi epoch, tự resume khi Colab ngắt)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token vào ô hiện ra.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()   # dán token vào ô hiện ra, rồi chạy tiếp cell Cấu hình
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 4. Cấu hình — bật/tắt module

In [ ]:
# ═══════════ BẬT / TẮT MODULE ═══════════
USE_QAVIT     = False   # True = đưa câu hỏi vào CLIP (QA-ViT) | False = CLIP thường
USE_SCESPAVIS = True    # True = SceSpaVis                      | False = OCR text-only
USE_OCRAUG    = True    # True = thêm bản OCR đã sửa (OCR-Aug)  | False = chỉ OCR gốc
DATASET       = "ViTextVQA"   # ViTextVQA | ViOCRVQA | OpenViVQA
RUN_TAG       = "r1"          # đổi khi muốn chạy lại từ đầu với repo HF mới
# ════════════════════════════════════════

import os
from huggingface_hub import get_token, HfApi

# finetune.py chỉ đọc HF_TOKEN từ biến môi trường → đưa token đã đăng nhập vào env
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

EPOCHS, BS, ACC, LR = 5, 4, 2, 3e-5
DATA_OUT = "./output/finetune_data"     # nơi finetune.py lưu/đọc CSV của dataset
TEST_GT_JSON = ""                       # (tuỳ chọn) đường dẫn JSON đáp án test để chấm offline

# Tên lần chạy tự sinh từ các module đang bật: "base", "scespavis", "scespavis-ocraug", ...
RUN_NAME = "-".join(m for m, on in (("qavit", USE_QAVIT), ("scespavis", USE_SCESPAVIS),
                                    ("ocraug", USE_OCRAUG)) if on) or "base"
OUT_DIR = f"./output/abl_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-abl-{DATASET.lower()}-{RUN_NAME}-{RUN_TAG}"
_main = (not USE_QAVIT) and USE_SCESPAVIS and USE_OCRAUG
print(f"Cấu hình: QA-ViT={USE_QAVIT} | SceSpaVis={USE_SCESPAVIS} | OCR-Aug={USE_OCRAUG}"
      f"{'   ← ViSceT5 (kết quả chính)' if _main else ''}")
print(f"RUN_NAME={RUN_NAME} | OUT_DIR={OUT_DIR} | HF_REPO={HF_REPO}")

## 5. Finetune (từ đầu, không pretrain)
`finetune.py` tự tải và chuẩn bị dữ liệu nếu chưa có, đẩy từng checkpoint lên `HF_REPO` và tự resume khi chạy lại. Xong, model tốt nhất (theo EM val) nằm ở `OUT_DIR` và được upload lên repo.

In [ ]:
import gc, torch
from huggingface_hub import HfApi

env = (f"OUTPUT_PATH={DATA_OUT} HF_REPO={HF_REPO} RESUME_FROM_HF=auto "
       f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")
args = ["configs/finetune.yaml",
        "--dataset_name", DATASET,
        "--ablation_use_vs", "False",
        "--ablation_use_qaclip", str(USE_QAVIT),
        "--ablation_use_ocr", str(USE_SCESPAVIS),
        "--ablation_use_ocr_aug", str(USE_OCRAUG),
        "--num_train_epochs", str(EPOCHS),
        "--per_device_train_batch_size", str(BS),
        "--per_device_eval_batch_size", str(BS),
        "--gradient_accumulation_steps", str(ACC),
        "--learning_rate", str(LR),
        "--bf16", "True",
        "--output_dir", OUT_DIR,
        "--logging_dir", OUT_DIR + "/logs"]
print(f"▶ {env} python training/finetune.py {' '.join(args)}")
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
get_ipython().system(env + " python training/finetune.py " + " ".join(args))

# Upload model tốt nhất (finetune.py chỉ tự push checkpoint-*)
if os.environ.get("HF_TOKEN") and os.path.exists(os.path.join(OUT_DIR, "model.safetensors")):
    try:
        api = HfApi(token=os.environ["HF_TOKEN"])
        api.create_repo(HF_REPO, repo_type="model", exist_ok=True, private=True)
        api.upload_folder(folder_path=OUT_DIR, repo_id=HF_REPO, repo_type="model",
                          ignore_patterns=["checkpoint-*/**", "*optimizer.pt", "*rng_state*", "*scheduler.pt"])
        print("☁️ Đã upload model tốt nhất →", HF_REPO)
    except Exception as e:
        print("⚠️ Upload lỗi:", e)

## 6. Dự đoán dev + test và chấm điểm
`predict.py` sinh câu trả lời (beam 4) cho dev (in EM/F1) và test (`submission_*.csv`). Nếu có file đáp án test (`TEST_GT_JSON`), `eval_submission.py` chấm EM, F1, BLEU, CIDEr.

In [ ]:
import os
if not os.path.exists(os.path.join(OUT_DIR, "model.safetensors")):
    print(f"⚠️ Chưa có model trong {OUT_DIR} — chạy cell Finetune trước.")
else:
    get_ipython().system(f"OUTPUT_PATH={DATA_OUT} python -m training.predict --split both --ckpt_dir {OUT_DIR} "
                         f"--dataset {DATASET} --batch_size 8 --num_beams 4 --out_dir {OUT_DIR}")
    tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
    pred = os.path.join(OUT_DIR, f"submission_{tag}test.csv")
    if TEST_GT_JSON and os.path.exists(TEST_GT_JSON) and os.path.exists(pred):
        get_ipython().system(f"python -m training.eval_submission --gt {TEST_GT_JSON} --pred {pred} "
                             f"--out {OUT_DIR}/eval_test.json")
    else:
        print(f"ℹ️ Chưa đặt TEST_GT_JSON → chỉ sinh {pred}, chưa chấm test.")

## 7. Tổng hợp kết quả mọi cấu hình đã chạy (trong `./output`)

In [ ]:
import json, os, glob
import pandas as pd
rows = []
for od in sorted(glob.glob(f"./output/abl_{DATASET}_*")):
    name = od.split(f"abl_{DATASET}_")[-1]
    mods = set(name.split("-"))
    r = {"run": name, "QA-ViT": "qavit" in mods, "SceSpaVis": "scespavis" in mods, "OCR-Aug": "ocraug" in mods}
    st = sorted(glob.glob(os.path.join(od, "checkpoint-*", "trainer_state.json")),
                key=lambda p: int(p.split("checkpoint-")[-1].split(os.sep)[0]))
    if st:
        evs = [h for h in json.load(open(st[-1])).get("log_history", []) if "eval_em" in h]
        if evs:
            best = max(evs, key=lambda h: h["eval_em"])
            r.update({"val_EM": round(100 * best["eval_em"], 2), "val_F1": round(100 * best.get("eval_f1", 0), 2),
                      "val_CIDEr": round(best.get("eval_cider", float("nan")), 4), "best_epoch": best.get("epoch")})
    ev = os.path.join(od, "eval_test.json")
    if os.path.exists(ev):
        r.update({f"test_{k}": v for k, v in json.load(open(ev)).items() if not isinstance(v, (dict, list))})
    rows.append(r)
df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "Chưa có cấu hình nào chạy xong.")